# EMNIST FPGA Live Classification — CNN Development and Camera Integration

This notebook documents the **implementation workflow** that adapted the Python-trained EMNIST Balanced compact CNN into an FPGA inference design, verified it with a fixed reference image, and integrated it with the existing OV7670 camera system on the ZedBoard Rev E.

It follows the model-development and implementation record rather than presenting a new training experiment. The camera acquisition and VGA subsystems were already developed and documented in the MNIST FPGA work; this notebook concentrates on **what changed for EMNIST**.

The final camera implementation provides **functional proof of concept**, not a controlled camera-classification accuracy estimate. Resource utilization, inference latency, throughput, timing-performance analysis, and power measurements belong in separate measurement notebooks.

## Scope

1. Relationship to the selected Python model and integer reference
2. Fixed-point parameters and layer-by-layer Verilog implementation
3. Historical Vivado behavioral simulations, including corrections
4. Complete CNN simulation and standalone ZedBoard verification
5. Successful standalone and integrated design routing
6. Reuse of the working MNIST OV7670 camera pipeline
7. Replacement of fixed-image ROM with camera-writable image memory
8. Restored thresholding, stored thumbnail, and live classification demonstrations
9. Preserved evidence, limitations, and reproducibility notes

**Related branches:** `EMNIST-Models`, `EMNIST-Models-Code`, `EMNIST-Models-Data-and-Notebook`, and `EMNIST-FPGA-Live-Classification-Code`.

## Files expected alongside this notebook

The GitHub branch stores evidence at its root. The notebook uses the **existing filenames**, including `.JPG` capitalization and compressed `.mp4` videos. No reorganization is required.

**Simulation logs:** `model_test_simulation_run1.txt`, `model_test2_simulation_run1.txt` through `model_test10_simulation_run1.txt` (as available), `model_test_11_simulation_run1.txt`, `model_test_12_simulation_run1.txt`, `model_test13_simulation_run1.txt`, `model_test13_simulation_run2.txt`, `model_test4_simulation_run2.txt`, and `model_test_board_simulation_run1.txt`.

**Implementation and hardware evidence:** `emnist_score_verification.ila`, `emnist_zedboard_reve_demo_top_route_status.rpt`, `ov7670_top_route_status.rpt`, `emnist_integer_comparison.png`.

**Character photos:** `emnist_2_live_classification.JPG`, `emnist_7_live_classification.JPG`, `emnist_F_live_classification.JPG`, `emnist_v_live_classification.JPG`.

**Compressed videos** (preserve exact GitHub names):

- `Emnist 2 Live Classification V11-16-202609-36.mp4`
- `Emnist 7 Live Classification V11-16-202609-36.mp4`
- `Emnist F Live Classification V11-16-202609-37.mp4`
- `Emnist V Live Classification V11-16-202609-50.mp4`

The video recordings were converted from the original MOV format to compressed MP4 files for repository storage. The notebook links to the MP4 files; it does not embed their full contents.

> **Scope note:** Model parameter `.mem` files, VHDL, Verilog, block-memory IP configuration, constraints, and the bitstream are retained in the companion **code branch**, rather than copied into this notebook/data branch.

## 1. Reference model selected in Python

The model-development experiments used the **EMNIST Balanced** dataset (47 classes, 28×28 grayscale images). The compact CNN was selected after evaluating fully connected and larger convolutional models. Its Python architecture comprised three 3×3 convolutions with 16, 32, and 64 output channels, two 2×2 maximum-pooling operations, global average pooling, and a 47-output fully connected layer.

```text
28 × 28 × 1
    ↓ Conv1 (16 channels), ReLU
14 × 14 × 16 after MaxPool1
    ↓ Conv2 (32 channels), ReLU
7 × 7 × 32 after MaxPool2
    ↓ Conv3 (64 channels), ReLU
7 × 7 × 64
    ↓ Global Average Pooling
64 features
    ↓ FC47
47 scores → argmax class
```

The trained compact model contained **26,351 parameters**, and its independent software test-set result was **16,327 / 18,800 (86.85%)**. These are **Python reference-model results**, not results from the camera-based FPGA demonstration. Model training, validation, checkpoint selection, and the complete dataset comparisons are documented in `EMNIST-Models-Data-and-Notebook`.

The FPGA implementation used the selected compact model's trained parameters rather than retraining a separate CNN on the ZedBoard.

## 2. From Python weights to an integer arithmetic reference

The FPGA implementation required conversion of the model's floating-point weights and biases into a reproducible integer representation. The software integer-reference work established layer scales, quantized weights and biases, integer multiply-accumulate behavior, and activation requantization for hardware comparison.

The development files included layer-specific weight, bias, and scaling information. In the verified RTL, the convolution scaling multipliers were represented as fixed Q24 constants in the arithmetic/controller logic. The parameter memories contained the quantized convolution and fully connected layer data.

**Model parameter files in the code branch:**

| Network stage | Parameter files |
|---|---|
| Conv1 | `weights.mem`, `bias.mem` |
| Conv2 | `weights2.mem`, `bias2.mem` |
| Conv3 | `weights3.mem`, `bias3.mem` |
| FC47 | `fc_weights.mem`, `fc_bias.mem` |

The integer-reference scripts and preliminary comparison were used to guide hardware adaptation. **A software integer-reference comparison is not proof that the physical FPGA computes the same values**; the later simulation and standalone board checks supplied additional verification.

### Integer-reference comparison evidence

![EMNIST software integer-reference comparison](emnist_integer_comparison.png)

This retained figure documents the model's software integer-reference comparison performed before and during the FPGA adaptation. Interpret the figure in conjunction with the original scripts; it does not report measured FPGA latency, power, or hardware accuracy.

## 3. Incremental CNN construction in Vivado

The compact CNN was constructed and simulated incrementally rather than integrating the complete network in a single step. Separate models tested convolution arithmetic, synchronous memory access, pooling, intermediate feature-map storage, global average pooling, and final classification.

The surviving Vivado console transcripts preserve both **successful validations** and **development errors**. In particular, earlier runs revealed problems such as unavailable memory-initialization files and watchdog timeouts; these should not be described as successful tests merely because later output appears in the same transcript.

The principal progression was:

```text
Conv1 arithmetic and BRAM
    ↓
Conv1 → MaxPool1
    ↓
Conv2 and integrated Conv1 → Pool1 → Conv2
    ↓
MaxPool2
    ↓
Conv3
    ↓
Global Average Pooling
    ↓
FC47 class scores and argmax
    ↓
Complete CNN
    ↓
Standalone ZedBoard demonstration
```

The Verilog modules and the final network hierarchy are preserved in `EMNIST-FPGA-Live-Classification-Code`.

### Reading the retained Vivado simulation logs

The following cell summarizes final PASS/FAIL markers found in each uploaded `.txt` transcript. It deliberately shows both markers when a log contains multiple development attempts. The source logs remain authoritative; this cell does not rerun Vivado simulations.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
logs = sorted(ROOT.glob("model_test*simulation*.txt"), key=lambda p: p.name.lower())
print(f"Found {len(logs)} simulation transcripts in {ROOT}")

for path in logs:
    text = path.read_text(encoding="utf-8", errors="replace")
    events = [line.strip() for line in text.splitlines()
              if "PASS:" in line or "FAIL:" in line or "FATAL:" in line]
    print(f"\n{path.name}")
    if events:
        for entry in events[-4:]:
            print("  " + (entry[:210] + "..." if len(entry) > 210 else entry))
    else:
        print("  No explicit PASS/FAIL/FATAL marker; inspect the transcript directly.")

### Selected simulation milestones

The following milestones were identified in the retained Vivado transcripts:

| Evidence file | What was verified or observed |
|---|---|
| `model_test2_simulation_run1.txt` | Initial Conv1 attempt included missing/unreadable memory-file warnings; not a clean initial validation |
| `model_test3_simulation_run1.txt` | 12,544 Conv1 outputs and BRAM readbacks passed |
| `model_test4_simulation_run2.txt` | A watchdog timeout was recorded during development; do not omit the failure |
| `model_test5_simulation_run1.txt` | 6,272 Conv2 outputs and BRAM readbacks passed |
| `model_test6_simulation_run1.txt` | Integrated Conv1 → Pool1 → Conv2 passed |
| `model_test7_simulation_run1.txt` | Integrated pipeline through MaxPool2 passed |
| `model_test8_simulation_run1.txt` | 3,136 Conv3 outputs and BRAM readbacks passed |
| `model_test9_simulation_run1.txt` | Integrated pipeline through Conv3 passed |
| `model_test10_simulation_run1.txt` | 64 GAP outputs and BRAM readbacks passed |
| `model_test_11_simulation_run1.txt` | Full Conv1 → Pool1 → Conv2 → Pool2 → Conv3 → GAP chain passed |
| `model_test_12_simulation_run1.txt` | 47 FC class scores and score-memory readbacks passed |
| `model_test13_simulation_run1.txt` | Complete CNN passed with class 45 and final score 950 |
| `model_test13_simulation_run2.txt` | Transcript contains an earlier elaboration error followed by a successful complete-CNN simulation |
| `model_test_board_simulation_run1.txt` | Complete CNN board-oriented simulation reproduced the reference class and score |

These results document **functional verification against expected intermediate and final outputs** for the recorded test configuration. They do not measure the model's accuracy over the EMNIST test set.

### Inspect any original log in context

Choose a filename from the retained transcripts to display its final simulation messages. This cell is read-only.

In [ ]:
from pathlib import Path

log_name = "model_test13_simulation_run1.txt"  # Change to another retained filename
log_path = Path(log_name)
if log_path.is_file():
    rows = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(rows[-35:]))
else:
    print(f"Missing file: {log_name}")

## 4. Complete CNN validation using the fixed reference image

The standalone FPGA model originally loaded a fixed 28×28 image from `image.mem`. Behavioral simulation compared layer outputs and final scores against reference values generated during the model-conversion work.

In the final complete-CNN simulation (`model_test13_simulation_run1.txt`), the transcript reported:

```text
PASS: complete CNN Conv1 -> Pool1 -> Conv2 -> Pool2 -> Conv3 -> GAP -> FC47
predicted_class=45 score=950
PASS: 47/47 integrated signed score RAM readbacks
```

The repeated successful simulation in `model_test13_simulation_run2.txt` and the board-oriented simulation in `model_test_board_simulation_run1.txt` reported the same final class and score. The second ModelTest13 transcript also contains earlier elaboration errors; its **final successful run** is distinguished from those earlier attempts.

These tests established consistent processing of the **known fixed input**, not general classification accuracy. The same trained weights and arithmetic were subsequently carried into the camera-enabled implementation.

### Review the recorded complete-CNN result

This read-only cell extracts the final complete-CNN PASS statements directly from the retained simulation logs.

In [ ]:
from pathlib import Path

for filename in ["model_test13_simulation_run1.txt",
                 "model_test13_simulation_run2.txt",
                 "model_test_board_simulation_run1.txt"]:
    file = Path(filename)
    if not file.exists():
        print(f"{filename}: file missing")
        continue
    matches = [line.strip() for line in file.read_text(errors="replace").splitlines()
               if line.strip().startswith("PASS: complete CNN") or
                  line.strip().startswith("PASS: 47/47 integrated signed")]
    print(f"\n{filename}")
    for line in matches[-2:]:
        print(line)

## 5. Standalone ZedBoard verification

Before connecting the camera, a standalone demonstration placed the compact CNN on the ZedBoard Rev E. The design used the fixed reference image and showed the predicted class through board indicators. The retained ILA session provided a hardware-level observation of the classification-completion and score signals.

The previously recorded standalone verification showed **class 45** with a **signed score of 950** for the fixed reference image, consistent with the complete-CNN simulation. This was a single known-input verification, not a test-set accuracy measurement.

The captured debug-session file is:

- [`emnist_score_verification.ila`](emnist_score_verification.ila)

An `.ila` file is a Vivado hardware-debug capture intended to be opened with compatible Vivado tools; this notebook does **not** attempt to decode it as text.

**Important:** This debug capture belongs to the **earlier standalone CNN project**. The **final live-camera design did not include an ILA/debug core**.

## 6. Standalone CNN and live-camera implementation verification

The branch preserves Vivado design-route status reports from two separate configurations:

- `emnist_zedboard_reve_demo_top_route_status.rpt` — standalone fixed-image CNN demonstration
- `ov7670_top_route_status.rpt` — integrated live-camera CNN design

The route reports establish that each implementation had no unrouted nets or routing errors at the time of reporting. **Route completion alone does not establish timing closure, successful bitstream programming, or classification correctness.** Functional evidence is provided by simulations and board demonstrations; quantitative implementation measurements are documented elsewhere.

In [ ]:
from pathlib import Path
import re

for filename in ["emnist_zedboard_reve_demo_top_route_status.rpt",
                 "ov7670_top_route_status.rpt"]:
    path = Path(filename)
    print("\n" + filename)
    if not path.is_file():
        print("  Not found in current directory")
        continue
    content = path.read_text(errors="replace")
    for label in ["# of routable nets", "# of fully routed nets", "# of nets with routing errors"]:
        match = re.search(re.escape(label) + r"[^:]*:\s*(\d+)", content)
        print(f"  {label}: {match.group(1) if match else 'not located'}")

## 7. Transition from fixed-image inference to the OV7670 camera

The live implementation began with the already working OV7670 camera project used in the MNIST experiments. The camera configuration, pixel capture, VGA output, thumbnail display, capture button, and live/stored-image switch had already been established, so their original development is not repeated here.

The main EMNIST-specific integration change was to replace the CNN's fixed `image.mem` ROM input with a writable **784-byte CNN input memory**. Camera-derived 28×28 pixels were written to the CNN input memory, while a separate stored-image path continued supporting the VGA thumbnail. The CNN was started after the captured image was frozen, preventing the classifier from intentionally mixing pixels from successive captures.

```text
OV7670 camera
    ↓
Existing camera acquisition and region-of-interest selection
    ↓
28×28 prepared image
    ├── Stored thumbnail memory → VGA view
    └── CNN input memory → Conv1 → Pool1 → Conv2 → Pool2 → Conv3 → GAP → FC47
                                                   ↓
                                             Predicted class LEDs
```

The convolution, pooling, GAP, FC47, trained-parameter, and Q24 arithmetic functions were kept consistent with the verified standalone CNN. The new live-camera `ov7670_top.vhd` coordinated the camera capture and CNN control interfaces.

The working top-level and all synthesis dependencies are preserved in `EMNIST-FPGA-Live-Classification-Code`.

## 8. Image preparation: returning to the working MNIST method

The first EMNIST integration evaluated weighted grayscale conversion with and without inversion. The inverted representation produced a dark digit on a light background and incorrect trial classifications; removing inversion resulted in an unusable black thumbnail under the observed conditions.

The implementation was therefore revised to preserve the **known working MNIST camera preprocessing**, while leaving the new EMNIST CNN connected:

- A 112×112 region of interest beginning at camera column 24 and row 4 was sampled.
- Every fourth pixel in each direction was selected, producing a 28×28 image (point sampling, not averaging).
- The upper four bits of the captured RGB444 pixel were compared against the threshold `7`.
- Pixels above the threshold were assigned `0x7F` (127); other pixels were assigned `0x00`.
- The existing binary thumbnail display was preserved for visual checking.

The resulting thresholded 28×28 image was supplied to Conv1. This representation **differs from the grayscale images used during Python training**; it was used here to establish end-to-end functional integration, not as a claim of equivalent input distribution.

### Thresholding logic preserved from MNIST

```vhdl
if capture_data(11 downto 8) > x"7" then
    roi_data_reg <= x"7F";
else
    roi_data_reg <= x"00";
end if;
```

The region-of-interest selection and thumbnail presentation remained based on the prior MNIST camera implementation. Their original creation, wiring, and VGA testing are documented in the MNIST FPGA branches.

## 9. Final camera classification interface

The integrated design retained the physical camera controls and presented the CNN output on the ZedBoard LEDs.

| Board signal | Function |
|---|---|
| Capture button | Freeze/release captured image |
| Live/stored switch | Select live view or stored thumbnail on VGA |
| LEDs 0–5 | Six-bit binary prediction for EMNIST classes 0–46 |
| LED 6 | CNN inference-completed indicator |
| LED 7 | Overflow diagnostic |

The six class LEDs are sufficient for 47 possible outputs. The output is a **numeric class ID**, interpreted using the EMNIST Balanced label mapping; it is not directly a printed character on the VGA display.

No cloud communication was involved, and the final live-camera configuration did **not** use an ILA debug core.

## 10. Live classification: physical character demonstrations

The retained photos and MP4 recordings document four informal physical-image trials. The images were manually positioned before the OV7670 camera; lighting and placement were not standardized. The captured thumbnail was used to inspect the prepared input, and the LEDs displayed the predicted class.

The reported observations were:

| Presented character | LEDs lit | Predicted class | Interpretation |
|---|---|---:|---|
| `2` | 6, 1 | 2 | Digit classified correctly; LED 6 indicated completion |
| `7` | 6, 2, 1, 0 | 7 | Digit classified correctly; LED 6 indicated completion |
| `F` | Not independently transcribed | 15 | Letter F was subsequently reported correctly classified |
| `v` | 6, 4, 1, 0 | 19 | Incorrect class for the intended letter |

**Evidence limitation:** The precise LED pattern for the successful `F` trial was not independently recorded in the conversation; the photograph/video are the authoritative visual record. The recorded trial for `T` also produced class 19, but a separate `T` recording was not among the four uploaded media examples.

These trials show that the integrated camera-to-CNN processing path operated and could classify several physical inputs. They do **not** constitute a standardized classification dataset or a percentage accuracy experiment.

### Digit 2 — classification demonstration

![Physical digit 2 and ZedBoard LEDs](emnist_2_live_classification.JPG)

[Open compressed MP4 recording — digit 2](Emnist%202%20Live%20Classification%20V11-16-202609-36.mp4)

The capture produced class ID **2**, with LED 6 indicating completion.

### Digit 7 — classification demonstration

![Physical digit 7 and ZedBoard LEDs](emnist_7_live_classification.JPG)

[Open compressed MP4 recording — digit 7](Emnist%207%20Live%20Classification%20V11-16-202609-36.mp4)

The capture produced class ID **7**, with LED 6 indicating completion.

### Letter F — classification demonstration

![Physical letter F and ZedBoard LEDs](emnist_F_live_classification.JPG)

[Open compressed MP4 recording — letter F](Emnist%20F%20Live%20Classification%20V11-16-202609-37.mp4)

A later observation identified **class 15 (F)** as correctly classified. The original photograph and video should be used to inspect the actual indicator state.

### Letter v — classification demonstration

![Physical letter v and ZedBoard LEDs](emnist_v_live_classification.JPG)

[Open compressed MP4 recording — letter v](Emnist%20V%20Live%20Classification%20V11-16-202609-50.mp4)

The reported LEDs 6, 4, 1, and 0 encode class ID **19** and completion; the intended `v` was not correctly recognized in this trial. Lighting, placement, thresholding, and handwritten-versus-printed character differences were possible influences, but the trial alone does not identify a single cause.

### Verify that supporting media is present

The compressed videos are linked using the filenames displayed in the GitHub uploads. If your local filenames or GitHub capitalization differ, update the corresponding Markdown link; GitHub paths are case-sensitive.

This cell checks the presence of the evidence files but does not analyze or alter their content.

In [ ]:
from pathlib import Path

media = [
    "emnist_2_live_classification.JPG",
    "emnist_7_live_classification.JPG",
    "emnist_F_live_classification.JPG",
    "emnist_v_live_classification.JPG",
    "Emnist 2 Live Classification V11-16-202609-36.mp4",
    "Emnist 7 Live Classification V11-16-202609-36.mp4",
    "Emnist F Live Classification V11-16-202609-37.mp4",
    "Emnist V Live Classification V11-16-202609-50.mp4",
]
for name in media:
    path = Path(name)
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {name}")

## 11. Implementation verification and evidence boundaries

The final live-camera project routed **4,318 / 4,318 routable nets with zero routing errors**, as recorded in `ov7670_top_route_status.rpt`. The earlier standalone ZedBoard demo routed **6,654 / 6,654** routable nets with zero routing errors, as recorded in `emnist_zedboard_reve_demo_top_route_status.rpt`.

These reports are retained as **build-completion evidence**. They are **not resource-utilization reports** and do not substitute for a separate timing analysis. The standalone ILA and behavioral simulations provide different kinds of functional verification; the live videos and photographs demonstrate physical camera acquisition and classification behavior.

The completed live-camera bitstream and its source hierarchy belong to the **code branch**, while the quantitative resource-utilization, latency, throughput, and power measurements will be documented separately.

## 12. What changed from MNIST, and what did not

| Area | EMNIST integration treatment |
|---|---|
| OV7670 initialization, image acquisition, clocking, VGA | Reused the established working MNIST camera implementation |
| Stored thumbnail and live/stored display switch | Preserved |
| 112×112 sampling region and 28×28 downsampling | Preserved |
| Pixel threshold | Restored the working MNIST red-channel threshold and `0x00`/`0x7F` values |
| NN architecture | Replaced the MNIST dense network with compact EMNIST CNN (16–32–64–47) |
| Input image source | Changed from fixed ROM image to camera-writable input memory |
| Learned parameters | Used selected EMNIST compact CNN weights and biases |
| Classification output | Expanded to a six-bit class ID for 47 classes |
| Hardware debug core | Present in earlier standalone verification, absent from final live camera project |

The implementation preserved the functioning camera pipeline and concentrated new RTL changes on the CNN input interface, class-output presentation, and integration control.

## 13. Final artifacts and reproducibility notes

| Artifact | Location / role |
|---|---|
| Selected compact CNN checkpoint and model summary | `EMNIST-Models` and model notebook/data branch |
| Original Python training and integer reference scripts | `EMNIST-Models-Code` and relevant model/FPGA development materials |
| Verilog CNN modules and camera VHDL sources | `EMNIST-FPGA-Live-Classification-Code` |
| Quantized `.mem` parameter files | `EMNIST-FPGA-Live-Classification-Code` |
| `.xci` memory IP and `.xdc` pin constraints | `EMNIST-FPGA-Live-Classification-Code` |
| Working live-camera bitstream | `EMNIST-FPGA-Live-Classification-Code` |
| Simulation transcripts | This notebook/data branch |
| Standalone `.ila` capture | This notebook/data branch; not from live design |
| Separate standalone and live routing reports | This notebook/data branch |
| Class-image photographs and compressed MP4 demonstrations | This notebook/data branch |

### Reproducibility limits

- The fixed-image behavioral simulation and standalone ILA verification concern a **single known reference image**.
- The successful live digit and letter demonstrations establish **functional proof of concept**, not camera-test-set accuracy.
- Physical cutouts were acquired under variable illumination and positioning, without standardized capture conditions.
- Restored binary thresholding differs from grayscale preprocessing used in the reference Python training pipeline.
- The full camera-system development is documented in the MNIST FPGA implementation branches rather than repeated here.
- Route-status reports establish routing completion but not implementation timing or power.
- The notebook **documents existing results**; it does not recreate the Vivado project, reprogram the FPGA, or generate new hardware measurements.

**Stopping point:** The compact EMNIST CNN was developed and verified as a standalone FPGA design, connected to the preexisting OV7670 capture pipeline, and shown to classify selected physical characters. Controlled performance benchmarking follows as a separate experiment.